In [ ]:
%pip install -q "monai==1.5.2" nibabel einops tqdm

In [ ]:
import csv
import json
import math
import random
import time
from pathlib import Path

import nibabel as nib
import numpy as np
import torch
import monai
from tqdm.auto import tqdm
from monai.data import DataLoader, Dataset
from monai.inferers import sliding_window_inference
from monai.losses import DiceLoss, FocalLoss, SoftclDiceLoss
from monai.networks.nets import SwinUNETR
from monai.transforms import (
    CastToTyped,
    Compose,
    EnsureChannelFirstd,
    Lambdad,
    LoadImaged,
    Rand3DElasticd,
    RandAdjustContrastd,
    RandAffined,
    RandBiasFieldd,
    RandCropByLabelClassesd,
    RandFlipd,
    RandGaussianNoised,
    RandGaussianSmoothd,
    RandScaleIntensityd,
    ResizeWithPadOrCropd,
    SpatialPadd,
)
from torch.optim import AdamW

print("PyTorch:", torch.__version__, "| MONAI:", monai.__version__)

In [ ]:
DATA_ROOT = Path("data/vascular")
SPLIT_FILE = Path("data/splits.json")
OUTPUT_DIR = Path("outputs/vascular_swin_unetr")
IMAGE_SUFFIX = "_image.nii.gz"
LABEL_SUFFIX = "_label.nii.gz"

INPUT_REQUIREMENTS = (
    "Each subject folder DATA_ROOT/<subject_id> contains <subject_id>" + IMAGE_SUFFIX + ", a preprocessed "
    "contrast-enhanced T1-weighted MRI volume (RAS orientation, 0.6 mm isotropic voxels, skull-stripped, "
    "N4 bias field corrected, intensities min-max scaled to 0-255), and <subject_id>" + LABEL_SUFFIX + ", "
    "the reference label map (0 = background, 1 = artery, 2 = vein; any other value is treated as background). "
    "SPLIT_FILE is a JSON file with the keys train, validation, and test, each listing subject identifiers."
)

HYPERPARAMETER_TUNING_SEED = 42
FINAL_TRAINING_SEED = 123
SEED = FINAL_TRAINING_SEED

TARGET_SPACING = (0.6, 0.6, 0.6)
NUM_CLASSES = 3
CLASS_LABELS = {"artery": 1, "vein": 2}

PATCH_SIZE = (192, 192, 192)
CROP_CLASS_RATIOS = (1, 7, 1)
SAMPLES_PER_VOLUME = 3
VOLUMES_PER_BATCH = 3
MICRO_BATCH_SIZE = 3

FEATURE_SIZE = 36
USE_SWIN_V2 = True
DROP_RATE = 0.0
ATTENTION_DROP_RATE = 0.0
STOCHASTIC_DEPTH_RATE = 0.0
USE_GRADIENT_CHECKPOINTING = True

FOCAL_GAMMA = 2.0
LAMBDA_DICE = 1.0
LAMBDA_FOCAL = 1.0
FOCAL_WEIGHT_ARTERY = 2.20
FOCAL_WEIGHT_VEIN = 0.75
CLDICE_WEIGHT = 0.5
CLDICE_START_EPOCH = 30
CLDICE_RAMP_EPOCHS = 20
CLDICE_SKELETON_ITERATIONS = 3
CLDICE_SMOOTH = 1e-5

LEARNING_RATE = 8e-5
WEIGHT_DECAY = 5e-5
WARMUP_EPOCHS = 10
WARMUP_START_FACTOR = 0.10
COSINE_END_EPOCH = 140
MIN_LEARNING_RATE = 1e-6
MAX_EPOCHS = 150
GRADIENT_CLIP_NORM = 0.5
USE_MIXED_PRECISION = True

AUG_FLIP_PROB = 0.5
AUG_AFFINE_PROB = 0.35
AUG_ROTATE_RANGE = (0.17, 0.17, 0.17)
AUG_SCALE_RANGE = (0.10, 0.10, 0.10)
AUG_ELASTIC_PROB = 0.20
AUG_ELASTIC_SIGMA_RANGE = (5.0, 8.0)
AUG_ELASTIC_MAGNITUDE_RANGE = (50.0, 100.0)
AUG_SMOOTH_PROB = 0.15
AUG_SMOOTH_SIGMA_RANGE = (0.3, 0.8)
AUG_NOISE_PROB = 0.35
AUG_NOISE_MEAN = 0.0
AUG_NOISE_STD = 0.02
AUG_SCALE_INTENSITY_PROB = 0.30
AUG_SCALE_INTENSITY_FACTOR = 0.15
AUG_BIAS_FIELD_PROB = 0.20
AUG_BIAS_FIELD_COEFF_RANGE = (0.0, 0.1)
AUG_CONTRAST_PROB = 0.20
AUG_CONTRAST_GAMMA = (0.7, 1.5)

SLIDING_WINDOW_OVERLAP = 0.6
SLIDING_WINDOW_BATCH_VALIDATION = 1
SLIDING_WINDOW_BATCH_INFERENCE = 4

NUM_WORKERS = 8
RESUME_FROM_LATEST_CHECKPOINT = False

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_split(split_file):
    with open(split_file, "r", encoding="utf-8") as handle:
        split = json.load(handle)
    for key in ("train", "validation", "test"):
        if key not in split:
            raise KeyError(f"The split file must contain the key '{key}'.")
    train_ids, validation_ids, test_ids = set(split["train"]), set(split["validation"]), set(split["test"])
    if train_ids & validation_ids or train_ids & test_ids or validation_ids & test_ids:
        raise ValueError("The training, validation, and test cohorts must not overlap.")
    return split


def check_input_volume(path):
    volume = nib.load(path)
    spacing = tuple(float(value) for value in volume.header.get_zooms()[:3])
    if not np.allclose(spacing, TARGET_SPACING, atol=1e-3):
        raise ValueError(f"Expected {TARGET_SPACING} mm voxels, found {spacing} in {path}")
    orientation = nib.aff2axcodes(volume.affine)
    if orientation != ("R", "A", "S"):
        raise ValueError(f"Expected RAS orientation, found {orientation} in {path}")


def build_file_list(subject_ids, require_label=True):
    files = []
    for subject_id in subject_ids:
        folder = DATA_ROOT / subject_id
        item = {
            "subject_id": subject_id,
            "image": str(folder / f"{subject_id}{IMAGE_SUFFIX}"),
            "label": str(folder / f"{subject_id}{LABEL_SUFFIX}"),
        }
        if not Path(item["image"]).is_file():
            raise FileNotFoundError(item["image"])
        if require_label and not Path(item["label"]).is_file():
            raise FileNotFoundError(item["label"])
        check_input_volume(item["image"])
        files.append(item)
    return files


set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
mixed_precision = USE_MIXED_PRECISION and device.type == "cuda"
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(INPUT_REQUIREMENTS)
split = load_split(SPLIT_FILE)
train_files = build_file_list(split["train"])
validation_files = build_file_list(split["validation"])
test_files = build_file_list(split["test"], require_label=False)
print("Training:", len(train_files), "| Validation:", len(validation_files), "| Test:", len(test_files))
print("Device:", device, "| Seed:", SEED)

In [ ]:
def keep_vessel_labels(label):
    if isinstance(label, torch.Tensor):
        cleaned = label.clone()
        cleaned[(cleaned != CLASS_LABELS["artery"]) & (cleaned != CLASS_LABELS["vein"])] = 0
        return cleaned
    cleaned = np.asarray(label).copy()
    cleaned[(cleaned != CLASS_LABELS["artery"]) & (cleaned != CLASS_LABELS["vein"])] = 0
    return cleaned


train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys=["label"], func=keep_vessel_labels),
    SpatialPadd(keys=["image", "label"], spatial_size=PATCH_SIZE),
    RandCropByLabelClassesd(
        keys=["image", "label"],
        label_key="label",
        spatial_size=PATCH_SIZE,
        ratios=list(CROP_CLASS_RATIOS),
        num_classes=NUM_CLASSES,
        num_samples=SAMPLES_PER_VOLUME,
        allow_smaller=False,
    ),
    ResizeWithPadOrCropd(keys=["image", "label"], spatial_size=PATCH_SIZE),
    RandFlipd(keys=["image", "label"], prob=AUG_FLIP_PROB, spatial_axis=[0, 1, 2]),
    RandAffined(
        keys=["image", "label"],
        prob=AUG_AFFINE_PROB,
        rotate_range=AUG_ROTATE_RANGE,
        scale_range=AUG_SCALE_RANGE,
        mode=("bilinear", "nearest"),
        padding_mode="zeros",
    ),
    Rand3DElasticd(
        keys=["image", "label"],
        prob=AUG_ELASTIC_PROB,
        sigma_range=AUG_ELASTIC_SIGMA_RANGE,
        magnitude_range=AUG_ELASTIC_MAGNITUDE_RANGE,
        mode=("bilinear", "nearest"),
        padding_mode="zeros",
    ),
    RandGaussianSmoothd(
        keys=["image"],
        prob=AUG_SMOOTH_PROB,
        sigma_x=AUG_SMOOTH_SIGMA_RANGE,
        sigma_y=AUG_SMOOTH_SIGMA_RANGE,
        sigma_z=AUG_SMOOTH_SIGMA_RANGE,
    ),
    RandGaussianNoised(keys=["image"], prob=AUG_NOISE_PROB, mean=AUG_NOISE_MEAN, std=AUG_NOISE_STD),
    RandScaleIntensityd(keys=["image"], factors=AUG_SCALE_INTENSITY_FACTOR, prob=AUG_SCALE_INTENSITY_PROB),
    RandBiasFieldd(keys=["image"], prob=AUG_BIAS_FIELD_PROB, coeff_range=AUG_BIAS_FIELD_COEFF_RANGE),
    RandAdjustContrastd(keys=["image"], prob=AUG_CONTRAST_PROB, gamma=AUG_CONTRAST_GAMMA),
    CastToTyped(keys=["image", "label"], dtype=(np.float32, np.int64)),
])

validation_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Lambdad(keys=["label"], func=keep_vessel_labels),
    CastToTyped(keys=["image", "label"], dtype=(np.float32, np.int64)),
])

inference_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    CastToTyped(keys=["image"], dtype=np.float32),
])

loader_options = {"num_workers": NUM_WORKERS, "pin_memory": device.type == "cuda"}
if NUM_WORKERS > 0:
    loader_options["persistent_workers"] = True
    loader_options["prefetch_factor"] = 4

train_loader = DataLoader(
    Dataset(data=train_files, transform=train_transforms),
    batch_size=VOLUMES_PER_BATCH,
    shuffle=True,
    **loader_options,
)
validation_loader = DataLoader(
    Dataset(data=validation_files, transform=validation_transforms),
    batch_size=1,
    shuffle=False,
    **loader_options,
)
print("Training batches per epoch:", len(train_loader), "| Validation volumes:", len(validation_loader))

In [ ]:
model = SwinUNETR(
    in_channels=1,
    out_channels=NUM_CLASSES,
    feature_size=FEATURE_SIZE,
    drop_rate=DROP_RATE,
    attn_drop_rate=ATTENTION_DROP_RATE,
    dropout_path_rate=STOCHASTIC_DEPTH_RATE,
    use_checkpoint=USE_GRADIENT_CHECKPOINTING,
    use_v2=USE_SWIN_V2,
).to(device)
trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"Swin UNETR trainable parameters: {trainable_parameters:,}")

In [ ]:
class VesselLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()
        focal_class_weights = torch.tensor([FOCAL_WEIGHT_ARTERY, FOCAL_WEIGHT_VEIN], dtype=torch.float32, device=device)
        self.dice = DiceLoss(include_background=False, to_onehot_y=True, softmax=True)
        self.focal = FocalLoss(include_background=False, to_onehot_y=True, gamma=FOCAL_GAMMA, weight=focal_class_weights)
        self.soft_cldice = SoftclDiceLoss(iter_=CLDICE_SKELETON_ITERATIONS, smooth=CLDICE_SMOOTH)
        self.alpha = 0.0
        self.last_dice_focal = float("nan")
        self.last_cldice = float("nan")

    def dice_focal_term(self, logits, labels):
        return LAMBDA_DICE * self.dice(logits, labels) + LAMBDA_FOCAL * self.focal(logits, labels)

    def cldice_term(self, logits, labels):
        probabilities = torch.softmax(logits.float(), dim=1)
        target = labels.long()
        if target.dim() == logits.dim():
            target = target.squeeze(1)
        target = torch.nn.functional.one_hot(target, num_classes=NUM_CLASSES).movedim(-1, 1).to(probabilities.dtype)
        return self.soft_cldice(target.contiguous(), probabilities)

    def forward(self, logits, labels):
        dice_focal = self.dice_focal_term(logits, labels)
        self.last_dice_focal = float(dice_focal.detach().item())
        self.last_cldice = float("nan")
        if self.alpha <= 0.0:
            return dice_focal
        with torch.autocast(device_type=logits.device.type, enabled=False):
            cldice = self.cldice_term(logits, labels)
        if not bool(torch.isfinite(cldice)):
            return dice_focal
        self.last_cldice = float(cldice.detach().item())
        return dice_focal + self.alpha * cldice.to(dice_focal.dtype)


def learning_rate_for_epoch(epoch):
    if epoch <= WARMUP_EPOCHS:
        progress = (epoch - 1) / max(WARMUP_EPOCHS - 1, 1)
        return LEARNING_RATE * (WARMUP_START_FACTOR + (1.0 - WARMUP_START_FACTOR) * progress)
    if epoch <= COSINE_END_EPOCH:
        progress = (epoch - WARMUP_EPOCHS) / max(COSINE_END_EPOCH - WARMUP_EPOCHS, 1)
        return MIN_LEARNING_RATE + (LEARNING_RATE - MIN_LEARNING_RATE) * 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LEARNING_RATE


def cldice_weight_for_epoch(epoch):
    if epoch < CLDICE_START_EPOCH:
        return 0.0
    if epoch >= CLDICE_START_EPOCH + CLDICE_RAMP_EPOCHS:
        return CLDICE_WEIGHT
    return CLDICE_WEIGHT * (epoch - CLDICE_START_EPOCH) / CLDICE_RAMP_EPOCHS


loss_function = VesselLoss().to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=mixed_precision)
CHECKPOINT_SELECTION_START_EPOCH = CLDICE_START_EPOCH + CLDICE_RAMP_EPOCHS

In [ ]:
metric_cldice = SoftclDiceLoss(iter_=CLDICE_SKELETON_ITERATIONS, smooth=CLDICE_SMOOTH)


def overlap_scores(prediction, reference):
    tp = float(torch.logical_and(prediction, reference).sum())
    fp = float(torch.logical_and(prediction, ~reference).sum())
    fn = float(torch.logical_and(~prediction, reference).sum())
    dice = 2.0 * tp / (2.0 * tp + fp + fn) if (2.0 * tp + fp + fn) > 0 else float("nan")
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    return dice, precision, recall


def cldice_score(prediction, reference):
    if not bool(prediction.any()) and not bool(reference.any()):
        return float("nan")
    predicted = prediction.float()[None, None]
    target = reference.float()[None, None]
    background = torch.zeros_like(predicted)
    loss = metric_cldice(torch.cat([background, target], dim=1), torch.cat([background, predicted], dim=1))
    value = 1.0 - float(loss)
    return min(max(value, 0.0), 1.0) if math.isfinite(value) else float("nan")


def nan_mean(values):
    finite = [float(value) for value in values if value is not None and math.isfinite(float(value))]
    return float(np.mean(finite)) if finite else float("nan")


def case_metrics(logits, labels):
    prediction = torch.argmax(logits, dim=1)[0]
    reference = labels[0, 0].long()
    row = {}
    for name, value in CLASS_LABELS.items():
        predicted_class = prediction == value
        reference_class = reference == value
        dice, precision, recall = overlap_scores(predicted_class, reference_class)
        row[f"{name}_dice"] = dice
        row[f"{name}_precision"] = precision
        row[f"{name}_recall"] = recall
        row[f"{name}_cldice"] = cldice_score(predicted_class, reference_class)
    for metric in ("dice", "precision", "recall", "cldice"):
        row[f"mean_{metric}"] = nan_mean([row[f"{name}_{metric}"] for name in CLASS_LABELS])
    return row


METRIC_KEYS = [f"{name}_{metric}" for name in list(CLASS_LABELS) + ["mean"] for metric in ("dice", "precision", "recall", "cldice")]


def summarize_cases(rows):
    return {key: nan_mean([row[key] for row in rows]) for key in METRIC_KEYS}

In [ ]:
def train_one_epoch(epoch):
    model.train()
    batch_losses, dice_focal_losses, cldice_losses, patch_rows = [], [], [], []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} training", leave=False):
        inputs = batch["image"].to(device, non_blocking=True)
        labels = batch["label"].to(device, non_blocking=True)
        total_patches = int(inputs.shape[0])
        optimizer.zero_grad(set_to_none=True)
        batch_loss = 0.0
        for start in range(0, total_patches, MICRO_BATCH_SIZE):
            inputs_micro = inputs[start:start + MICRO_BATCH_SIZE]
            labels_micro = labels[start:start + MICRO_BATCH_SIZE]
            share = inputs_micro.shape[0] / total_patches
            with torch.autocast(device_type=device.type, enabled=mixed_precision):
                logits = model(inputs_micro)
                loss = loss_function(logits, labels_micro)
            scaler.scale(loss * share).backward()
            batch_loss += float(loss.detach().item()) * share
            dice_focal_losses.append(loss_function.last_dice_focal)
            cldice_losses.append(loss_function.last_cldice)
            with torch.no_grad():
                for index in range(logits.shape[0]):
                    patch_rows.append(case_metrics(logits[index:index + 1].float(), labels_micro[index:index + 1]))
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        batch_losses.append(batch_loss)
    summary = summarize_cases(patch_rows)
    summary["loss"] = nan_mean(batch_losses)
    summary["loss_dice_focal"] = nan_mean(dice_focal_losses)
    summary["loss_cldice"] = nan_mean(cldice_losses)
    return summary


def validate(epoch):
    model.eval()
    losses, dice_focal_losses, cldice_losses, case_rows = [], [], [], []
    with torch.no_grad():
        for batch in tqdm(validation_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} validation", leave=False):
            inputs = batch["image"].to(device)
            labels = batch["label"].to(device)
            with torch.autocast(device_type=device.type, enabled=mixed_precision):
                logits = sliding_window_inference(
                    inputs,
                    roi_size=PATCH_SIZE,
                    sw_batch_size=SLIDING_WINDOW_BATCH_VALIDATION,
                    predictor=model,
                    overlap=SLIDING_WINDOW_OVERLAP,
                )
            losses.append(float(loss_function(logits, labels).item()))
            dice_focal_losses.append(loss_function.last_dice_focal)
            cldice_losses.append(loss_function.last_cldice)
            row = case_metrics(logits.float(), labels)
            row["subject_id"] = batch["subject_id"][0]
            case_rows.append(row)
    summary = summarize_cases(case_rows)
    summary["loss"] = nan_mean(losses)
    summary["loss_dice_focal"] = nan_mean(dice_focal_losses)
    summary["loss_cldice"] = nan_mean(cldice_losses)
    return summary, case_rows

In [ ]:
HISTORY_FILE = OUTPUT_DIR / "training_history.csv"
VALIDATION_CASES_FILE = OUTPUT_DIR / "validation_cases.csv"
LATEST_CHECKPOINT_FILE = OUTPUT_DIR / "latest_checkpoint.pt"
BEST_MODEL_FILE = OUTPUT_DIR / "best_model.pt"
SUMMARY_KEYS = ["loss", "loss_dice_focal", "loss_cldice"] + METRIC_KEYS
HISTORY_FIELDS = ["epoch", "learning_rate", "cldice_weight"] + [f"train_{key}" for key in SUMMARY_KEYS] + [f"validation_{key}" for key in SUMMARY_KEYS]
CASE_FIELDS = ["epoch", "subject_id"] + METRIC_KEYS


def append_rows(path, fields, rows):
    write_header = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)


start_epoch = 1
best_validation_loss = float("inf")
best_epoch = None
if RESUME_FROM_LATEST_CHECKPOINT and LATEST_CHECKPOINT_FILE.is_file():
    state = torch.load(LATEST_CHECKPOINT_FILE, map_location=device)
    model.load_state_dict(state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scaler.load_state_dict(state["scaler"])
    start_epoch = int(state["epoch"]) + 1
    best_validation_loss = float(state["best_validation_loss"])
    best_epoch = state["best_epoch"]
    print("Resuming from epoch", start_epoch)

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    started = time.time()
    learning_rate = learning_rate_for_epoch(epoch)
    for group in optimizer.param_groups:
        group["lr"] = learning_rate
    loss_function.alpha = cldice_weight_for_epoch(epoch)

    train_summary = train_one_epoch(epoch)
    validation_summary, validation_cases = validate(epoch)

    if epoch >= CHECKPOINT_SELECTION_START_EPOCH and validation_summary["loss"] < best_validation_loss:
        best_validation_loss = validation_summary["loss"]
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_FILE)

    history_row = {"epoch": epoch, "learning_rate": learning_rate, "cldice_weight": loss_function.alpha}
    history_row.update({f"train_{key}": train_summary[key] for key in SUMMARY_KEYS})
    history_row.update({f"validation_{key}": validation_summary[key] for key in SUMMARY_KEYS})
    append_rows(HISTORY_FILE, HISTORY_FIELDS, [history_row])
    append_rows(VALIDATION_CASES_FILE, CASE_FIELDS, [{"epoch": epoch, **row} for row in validation_cases])

    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict(),
            "best_validation_loss": best_validation_loss,
            "best_epoch": best_epoch,
        },
        LATEST_CHECKPOINT_FILE,
    )
    print(
        f"Epoch {epoch}/{MAX_EPOCHS} | lr {learning_rate:.2e} | clDice weight {loss_function.alpha:.3f} | "
        f"train loss {train_summary['loss']:.4f} | validation loss {validation_summary['loss']:.4f} | "
        f"validation DSC artery {validation_summary['artery_dice']:.4f} vein {validation_summary['vein_dice']:.4f} | "
        f"validation clDice artery {validation_summary['artery_cldice']:.4f} vein {validation_summary['vein_cldice']:.4f} | "
        f"{time.time() - started:.0f} s"
    )

print("Selected checkpoint: epoch", best_epoch, "| total validation loss", best_validation_loss)

In [ ]:
PREDICTION_DIR = OUTPUT_DIR / "test_predictions"
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
model.load_state_dict(torch.load(BEST_MODEL_FILE, map_location=device))
model.eval()
with torch.no_grad():
    for item in tqdm(test_files, desc="Test cohort inference"):
        data = inference_transforms({"image": item["image"]})
        inputs = data["image"].unsqueeze(0).to(device)
        with torch.autocast(device_type=device.type, enabled=mixed_precision):
            logits = sliding_window_inference(
                inputs,
                roi_size=PATCH_SIZE,
                sw_batch_size=SLIDING_WINDOW_BATCH_INFERENCE,
                predictor=model,
                overlap=SLIDING_WINDOW_OVERLAP,
            )
        prediction = torch.argmax(logits, dim=1)[0].cpu().numpy().astype(np.uint8)
        reference = nib.load(item["image"])
        nib.save(nib.Nifti1Image(prediction, reference.affine), str(PREDICTION_DIR / f"{item['subject_id']}_prediction.nii.gz"))
print("Test cohort predictions saved to", PREDICTION_DIR)